In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore


C:\Users\UseR\AppData\Local\Temp\ipykernel_20824\3113008636.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()
len(docs)

10

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

splitted_data = splitter.split_documents(docs)
len(splitted_data)

11

In [5]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [6]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db"
)

In [12]:
query = "AI/ML Engineer vs SWE"
data = vector_store.similarity_search(query=query)

data k context e convert

In [13]:
context = ""
for doc in data:
    context+=doc.page_content + "\n"

print(context)

🎤  Mock Interview: Stats Scenarios  Probabilities  Tests
✅  Module 6: Machine Learning – I (Supervised 
Learning) (4 weeks)
Duration: Month 6
Topics:
ML pipeline
Regression: Linear, Logistic
Decision Tree, Random Forest, KNN
Train-test split, model evaluation
Tools:
Scikit-learn, Google Colab, ChatGPT, PyCaret (optional)
Mini Project:
Loan Approval or House Price Prediction
Predict Diabetes from health dataset
🎤  Mock Interview: Supervised Learning Models  Metrics
✅  Module 7: Machine Learning – II (Unsupervised & 
Feature Engineering) (3 weeks)
Duration: Month 7
Topics:
KMeans Clustering
Dimensionality Reduction: PCA
Feature selection, encoding, scaling
Model tuning GridSearchCV
Tools:
Scikit-learn, Seaborn, Colab
Mini Project:
📘  1 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI
5
🎤  Mock Interview: Stats Scenarios  Probabilities  Tests
✅  Module 6: Machine Learning – I (Supervised 
Learning) (4 weeks)
Duration: Month 6
Topics:
ML pipeline
Regression: Linear, Logis

In [14]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [15]:
res = llm.invoke(f"""Can you provide me the answer based on the provided
                 context: {context} and question: {query}""")

print(res.content)

## AI/ML Engineer **vs** Software Engineer (SWE)

| Dimension | **AI/ML Engineer** | **Software Engineer (SWE)** |
|-----------|--------------------|-----------------------------|
| **Primary focus** | Building, training, validating, and deploying **machine‑learning models** and data‑centric pipelines. | Building, testing, deploying, and maintaining **software systems** (web apps, services, libraries, etc.). |
| **Typical day‑to‑day work** | • Data wrangling & feature engineering (pandas, SQL, Spark).  <br>• Experimenting with models (scikit‑learn, PyTorch, TensorFlow).  <br>• Hyper‑parameter tuning (GridSearchCV, Optuna).  <br>• Model evaluation (accuracy, ROC‑AUC, SHAP).  <br>• MLOps: versioning data & models (MLflow, DVC), CI/CD for models, monitoring drift. | • Designing system architecture (REST, gRPC, micro‑services).  <br>• Writing clean, testable code (Java, C#, Go, Python).  <br>• Building APIs, UI, or backend services.  <br>• Writing unit/integration tests, code reviews.  <br

### Rag using proper pipline which is chain

Chain hobe - context generate | prompt | llm | strparser

In [25]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context+=doc.page_content + "\n"

    return {
        "context": context,
        "question": query
    }
    

In [26]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant. Answer the user's question based only on the provided context.
    If the answer is not available in the context, say: "I don't know."
    Context: {context}
    Question: {question}
""")

In [27]:
rag_chain = get_context | prompt | llm

In [28]:
res = rag_chain.invoke("What is his name?")

In [30]:
print(res.content)

I don't know.
